# 07 · CP4 preparation: development split and BR-Gen (decisions of 15–16 Nov 2026)

**CPU runtime** (no GPU needed; about 0.08 compute units per hour).

1. **Development split (frozen).**
   - It is built from the CP1 run-5 manifest:
     - P1: a seeded 20% of each of the 16 GenImage validation folders;
     - P2: a seeded 20% of the held-out-editor images, stratified by base type.
   - Identical images always land on the same side.
   - It is saved once to `runs/checkpoints/cp4_dev_split.json`. Later runs **load and verify** it and never overwrite it.
   - No image is opened.
2. **BR-Gen archive completed, then listed.**
   - The split archive must be whole: `BR-Gen_split.z01`…`z12` and the last volume `BR-Gen_split.zip`, which holds the table of contents (Hugging Face `clpbc/BR-Gen`). On 17 Nov Drive held only `z01`…`z08`.
   - Cell 5 downloads only what is missing (about 25.5 GB on 17 Nov: `z09`–`z12` and the `.zip`) into `04_BR-Gen`, checks each new volume's size and SHA-256, and refuses up front if the Drive quota is short. Volumes already there with the published size are kept. Running it again downloads nothing.
   - `7z` then reads the table of contents without extracting any image and writes `runs/checkpoints/brgen_listing.json`.
3. **BR-Gen subset and sample check: only after Dhanush approves the subset design.** Set `FREEZE_BRGEN = True` and the subset settings in cell 2.
   - The subset is frozen by seed and file-list SHA-256 in `runs/checkpoints/brgen_subset.json`, and is never overwritten.
   - The sample check extracts about 20 images and their masks and checks:
     - decoding;
     - mask size;
     - full tile coverage;
     - that stitching gives the mask back;
     - the tiled path with an **untrained** model.
   - No trained model is scored.

**Results** (send to Claude):
- `Paper 1/runs/checkpoints/cp4_prep.json`
- `Paper 1/runs/checkpoints/brgen_listing.json`

In [ ]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: fake sources, temp folders
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'                                    # 'claude/nice-curie-3tzxzr' before the merge

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO + '[train]'], check=True)
    # Colab ships torchao 0.10; peft refuses a torchao older than 0.16 with an ImportError when it
    # adds the LoRA layers, although EviGround never uses torchao: remove an old one.
    import importlib, importlib.metadata as _md
    from packaging.version import Version
    try:
        if Version(_md.version('torchao')) < Version('0.16.0'):
            subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'], check=True)
            importlib.invalidate_caches()
    except _md.PackageNotFoundError:
        pass

sys.path.insert(0, os.path.join(REPO, 'src'))
# Re-running in a runtime that already imported eviground would keep the OLD code in memory
# although git pulled the new one: drop it so the next import reads the files on disk.
for _name in [m for m in sys.modules if m == 'eviground' or m.startswith('eviground.')]:
    del sys.modules[_name]
print('repo:', REPO, '| branch:', BRANCH, '| dry run:', DRY)

In [ ]:
# Cell 2: settings
import datetime, gzip, hashlib, json, logging, os, subprocess, time
from pathlib import Path
logging.basicConfig(level=logging.INFO, force=True)
from eviground.env import get_paths, session_info

CU_PER_HOUR = 0.08                 # CPU runtime: read the live rate in Colab's resource panel
FREEZE_BRGEN = False               # True only after Dhanush approves the BR-Gen subset design
SAMPLE_CHECK = False               # True together with FREEZE_BRGEN (or once the subset exists)
STRATUM_DEPTH = None               # BR-Gen subset: folders that define a stratum, plus the method prefix
PER_STRATUM = None                 # BR-Gen subset: images per stratum (approved value)
N_SAMPLE = 20                      # images extracted for the sample check

T0 = time.time()
info = session_info(cu_per_hour=CU_PER_HOUR)
paths = get_paths()
if DRY:   # fake world, fake BR-Gen archive
    import shutil, tempfile
    from eviground.env import Paths
    _dry = Path(tempfile.gettempdir()) / 'eviground_cp4prep_dry'
    shutil.rmtree(_dry, ignore_errors=True)
    paths = Paths(root=_dry / 'drive' / 'Paper 1', local_data=_dry / 'data')
    FREEZE_BRGEN, SAMPLE_CHECK, STRATUM_DEPTH, PER_STRATUM, N_SAMPLE = True, True, 2, 2, 4
paths.ensure()
CK = paths.runs / 'checkpoints'
MANIFEST = CK / 'cp1' / 'manifest_cp1.csv.gz'
DEV_FILE = CK / 'cp4_dev_split.json'
BRGEN_DIR = paths.datasets / '04_BR-Gen'
RESULT = {'step': 'CP4 preparation', 'dry_run': DRY, 'branch': BRANCH, 'session': info}
print('commit', info['git_commit'], '| manifest', MANIFEST, '| BR-Gen', BRGEN_DIR)

In [ ]:
# Cell 3: dry run only: a fake CP1 manifest and a small fake BR-Gen archive
if DRY:
    import io, zipfile
    import numpy as np
    from PIL import Image
    from eviground.data.cp1 import CP1Config, run_cp1
    from eviground.data.download import ensure_data
    from eviground.data.fake_cp1 import cp1_overrides, make_cp1_world
    sources = make_cp1_world(paths.datasets, paths.root.parent / 'fake_remote', coco_images=40)
    ensure_data('cp1', paths=paths, sources=sources, stage_local=True, overrides=cp1_overrides(40))
    run_cp1(paths, CP1Config(workers=2, crops_per_class=60, blockiness_per_group=20,
                             base_unclear_max=0.2, diffseg_card_splits={'train': 18, 'validation': 12}),
            out_dir=CK / 'cp1', cache_dir=paths.cache / 'cp1', progress=None)
    from eviground.data import brgen
    from eviground.data.fake_sources import FakeSources
    BRGEN_DIR.mkdir(parents=True, exist_ok=True)
    remote = paths.root.parent / 'fake_brgen_remote'   # stands in for Hugging Face clpbc/BR-Gen
    remote.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(0)
    def png(arr):
        buf = io.BytesIO(); Image.fromarray(arr).save(buf, 'PNG'); return buf.getvalue()
    with zipfile.ZipFile(remote / 'BR-Gen_split.zip', 'w') as z:   # BR-Gen's layout: Au/, Tp/, Gt/
        for region in ('Stuff', 'Background'):
            root = f'BR-Gen/BRGen-{region}-Val'
            for i in range(4):
                h, w = (300, 520) if i % 2 else (1024, 700)
                src = f'COCO_{i:011d}'
                z.writestr(f'{root}/Au/{src}.png', png(rng.integers(0, 255, (h, w, 3), dtype=np.uint8)))
                for method in ('MAT', 'SDXL'):
                    mask = np.zeros((h, w), np.uint8); mask[: h // 3, : w // 4] = 255
                    z.writestr(f'{root}/Tp/{method}_{region}_{src}.png', png(rng.integers(0, 255, (h, w, 3), dtype=np.uint8)))
                    z.writestr(f'{root}/Gt/{method}_{region}_{src}_mask.png', png(mask))
    BRGEN_SOURCES = FakeSources(hf={brgen.REPO: {'BR-Gen_split.zip': remote / 'BR-Gen_split.zip'}})
else:
    BRGEN_SOURCES = None   # Hugging Face
print('manifest present:', MANIFEST.is_file())

In [ ]:
# Cell 4: development split (frozen once; reused and verified afterwards)
from eviground.data.cp1 import _read_csv_gz
from eviground.eval import devsplit

rows = _read_csv_gz(MANIFEST)
if DEV_FILE.is_file():
    split = devsplit.load(DEV_FILE)
    check = devsplit.verify(split, rows)
    assert check['same'], f'the frozen dev split no longer matches the manifest: {check}'
    print('frozen split reused and verified:', split['dev_sha256'])
else:
    split = devsplit.build_dev_split(rows)
    devsplit.save(split, DEV_FILE)
    print('dev split frozen:', split['dev_sha256'])
RESULT['dev_split'] = {**devsplit.summary(split), 'file': str(DEV_FILE)}
print(json.dumps({k: v for k, v in RESULT['dev_split'].items() if k != 'strata'}, indent=1))
for s, c in split['strata'].items():
    print(f'  {s:42} {c["dev"]:>6} of {c["images"]:>6}')

In [ ]:
# Cell 5: complete the BR-Gen split archive on Drive (downloads only missing volumes)
from eviground.data import brgen, download

ctx = download.make_context(paths, BRGEN_SOURCES)
PARTS = brgen.complete_split(ctx, BRGEN_DIR)
RESULT['brgen_parts'] = PARTS
print(f"{len(PARTS['expected'])} volumes published; kept {len(PARTS['kept'])}, downloaded "
      f"{len(PARTS['downloaded'])} ({PARTS['bytes_downloaded'] / 1e9:.1f} GB) in {PARTS['seconds']:.0f} s")
for key in ('downloaded', 'replaced', 'size_only', 'extra'):
    if PARTS[key]:
        print(f'  {key}: {PARTS[key]}')

In [ ]:
# Cell 6: BR-Gen listing (table of contents only; no image extracted)
from collections import Counter
from eviground.data import brgen

exe = brgen.seven_zip()
if exe is None and not DRY:
    subprocess.run(['apt-get', 'install', '-y', '-q', 'p7zip-full'], check=True, capture_output=True)
    exe = brgen.seven_zip()
assert exe, '7z is not available'
parts = sorted(PARTS['expected'])
missing = [p for p in parts if not (BRGEN_DIR / p).is_file()]
assert not missing, f'BR-Gen volumes missing after cell 5: {missing}'
ARCHIVE = BRGEN_DIR / 'BR-Gen_split.zip'
t = time.time()
entries = brgen.list_archive(ARCHIVE, exe)
files = [e['path'] for e in entries if not e['is_dir']]
pairs = brgen.pair_masks(files)
LISTING = {
    'archive_parts': parts,
    'methods': dict(sorted(Counter(brgen.method_of(p) for p in files if brgen.kind(p) == 'image').items())),
    'list_seconds': round(time.time() - t, 1),
    **brgen.summarize(entries),
    'pairing': {k: v for k, v in pairs.items() if k != 'pairs'},
    'pairs': len(pairs['pairs']),
}
(CK / 'brgen_listing.json').write_text(json.dumps(LISTING, indent=1))
with gzip.open(CK / 'brgen_members.txt.gz', 'wt') as fh:
    fh.write('\n'.join(files) + '\n')
RESULT['brgen_listing'] = {k: LISTING[k] for k in ('archive_parts', 'files', 'bytes', 'kinds', 'methods', 'pairs', 'pairing')}
print(json.dumps(RESULT['brgen_listing'], indent=1))

In [ ]:
# Cell 7: BR-Gen subset (only after the design is approved; frozen once, never overwritten)
SUBSET_FILE = CK / 'brgen_subset.json'
SUBSET = None
if SUBSET_FILE.is_file():
    SUBSET = json.loads(SUBSET_FILE.read_text())
    from eviground.eval.devsplit import file_list_sha256
    assert file_list_sha256(SUBSET['items']) == SUBSET['sha256'], 'brgen_subset.json was changed'
    print('frozen subset reused:', SUBSET['sha256'], len(SUBSET['items']), 'images')
elif FREEZE_BRGEN:
    assert STRATUM_DEPTH and PER_STRATUM, 'set the approved STRATUM_DEPTH and PER_STRATUM in cell 2'
    stratum = lambda p: brgen.stratum_key(p, STRATUM_DEPTH)
    SUBSET = brgen.subset(sorted(pairs['pairs']), PER_STRATUM, stratum)
    SUBSET.update(stratum_depth=STRATUM_DEPTH, per_stratum=PER_STRATUM,
                  masks={i: pairs['pairs'][i] for i in SUBSET['items']})
    SUBSET_FILE.write_text(json.dumps(SUBSET, indent=1))
    print('subset frozen:', SUBSET['sha256'], len(SUBSET['items']), 'images')
else:
    print('BR-Gen subset not frozen yet (waiting for the approved design)')
if SUBSET:
    RESULT['brgen_subset'] = {k: SUBSET[k] for k in ('seed', 'salt', 'sha256', 'strata', 'stratum_depth', 'per_stratum')}
    RESULT['brgen_subset']['images'] = len(SUBSET['items'])

In [ ]:
# Cell 8: sample check (loading, masks, tiling, stitching; an UNTRAINED model, no scoring)
if SAMPLE_CHECK and SUBSET:
    import shutil, tempfile
    import numpy as np, torch
    from PIL import Image
    from eviground.config import load_config
    from eviground.data import tiles
    from eviground.models.eviground import build_model

    picks = brgen.sample(SUBSET['items'], N_SAMPLE)
    out_dir = Path(tempfile.mkdtemp(prefix='brgen_sample_'))
    t = time.time()
    brgen.extract(ARCHIVE, picks + [SUBSET['masks'][p] for p in picks], out_dir, exe)
    extract_s = time.time() - t
    model = build_model(load_config('debug').model).eval()   # untrained, tiny: path check only
    rows = []
    for p in picks:
        img = np.asarray(Image.open(out_dir / p).convert('RGB'))
        mask = np.asarray(Image.open(out_dir / SUBSET['masks'][p]).convert('L'))
        h, w = img.shape[:2]
        boxes = tiles.tile_boxes(h, w)
        m = torch.from_numpy(mask.astype(np.float32))
        back = tiles.stitch(torch.stack([m[t0:b, l:r] for l, t0, r, b in boxes]), boxes, h, w)
        x = torch.from_numpy(img).permute(2, 0, 1).float() / 255
        t1 = time.time()
        pred = model.predict_tiled(x)
        rows.append({
            'image': p, 'height': h, 'width': w, 'mask_same_size': mask.shape == (h, w),
            'mask_values': sorted(int(v) for v in np.unique(mask))[:6],
            'edited_share': round(float((mask > 127).mean()), 4),
            'tiles': len(boxes), 'coverage': tiles.coverage(h) * tiles.coverage(w),
            'stitch_identity': bool(torch.equal(back, m)),
            'pred_mask_shape_ok': tuple(pred['mask'].shape) == (h, w),
            'seconds_per_tile_cpu_tiny_model': round((time.time() - t1) / len(boxes), 4),
        })
    shutil.rmtree(out_dir, ignore_errors=True)
    ok = all(r['mask_same_size'] and abs(r['coverage'] - 1.0) < 1e-9 and r['stitch_identity']
             and r['pred_mask_shape_ok'] for r in rows)
    RESULT['brgen_sample_check'] = {
        'images': len(rows), 'extract_seconds': round(extract_s, 1), 'max_tiles': max(r['tiles'] for r in rows),
        'pass': ok, 'rows': rows,
        'note': 'untrained tiny model on CPU: checks the path only; no score, timing not representative',
    }
    (CK / 'brgen_sample_check.json').write_text(json.dumps(RESULT['brgen_sample_check'], indent=1))
    print('sample check pass:', ok, '| max tiles', RESULT['brgen_sample_check']['max_tiles'])
else:
    print('sample check skipped')

In [ ]:
# Cell 9: the result file (send cp4_prep.json and brgen_listing.json to Claude)
hours = (time.time() - T0) / 3600
RESULT.update({
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'wall_time_h': round(hours, 3), 'cu_per_hour': CU_PER_HOUR,
    'compute_units': round(hours * CU_PER_HOUR, 3),
})
out = CK / 'cp4_prep.json'
out.write_text(json.dumps(RESULT, indent=1, default=str))
print('saved', out, '| CU', RESULT['compute_units'])